In [ ]:

# CELL 1 — BASELINE: build a CONFIDENCE/ASSERTIVENESS model, run it, save the BASE model
#
# IMPORTANT:
# This notebook intentionally does NOT use emotion labels such as HAPPY, SAD, DISGUSTED, etc.
# The task is:
#   1) CONFIDENT      — primary score
#   2) ASSERTIVE      — optional secondary score
#
# Public speech-emotion datasets generally do not contain a ground-truth "confidence" label.
# Therefore this notebook uses MSP-Podcast's continuous DOMINANCE rating as a documented
# behavioral proxy for confidence/assertiveness. The output is a model score for the proxy,
# NOT a claim that dominance == confidence.
#
# We use a small pretrained wav2vec2 speech encoder and add two regression heads.
# A binary "CONFIDENT" decision is derived from the continuous confidence score.
# The BASE model is saved locally at the end of this cell.
#
# Required input dataset format if you already have labeled data:
#   audio_path, confidence_score, assertiveness_score
# where scores are in [0, 1].
#
# If you only have MSP-Podcast dominance:
#   confidence_score = dominance
#   assertiveness_score = dominance
# This is a proxy and should be replaced with human confidence/asssertiveness labels
# for a production system.

!pip -q install -U transformers datasets librosa soundfile scikit-learn pandas numpy scipy tqdm accelerate

import os, json, math, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import Wav2Vec2Model, Wav2Vec2Processor

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL_NAME = "facebook/wav2vec2-base"
BASE_SAVE_DIR = Path("./models/confidence_base")
TARGET_SR = 16000
MAX_SECONDS = 2.0
MAX_SAMPLES = int(TARGET_SR * MAX_SECONDS)

print("Device:", DEVICE)
print("Base encoder:", BASE_MODEL_NAME)

# ---------------------------------------------------------------------
# 1. MODEL
# ---------------------------------------------------------------------
processor = Wav2Vec2Processor.from_pretrained(BASE_MODEL_NAME)
encoder = Wav2Vec2Model.from_pretrained(BASE_MODEL_NAME).to(DEVICE)
encoder.eval()

class ConfidenceHead(nn.Module):
    """
    Two continuous outputs:
      confidence_score  in [0,1]
      assertiveness_score in [0,1]

    The final sigmoid makes the outputs directly interpretable as percentages.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.dropout = nn.Dropout(0.10)
        self.confidence = nn.Linear(hidden_size, 1)
        self.assertiveness = nn.Linear(hidden_size, 1)

    def forward(self, pooled):
        x = self.dropout(pooled)
        confidence = torch.sigmoid(self.confidence(x)).squeeze(-1)
        assertiveness = torch.sigmoid(self.assertiveness(x)).squeeze(-1)
        return confidence, assertiveness

head = ConfidenceHead(encoder.config.hidden_size).to(DEVICE)

# ---------------------------------------------------------------------
# 2. AUDIO + INFERENCE HELPERS
# ---------------------------------------------------------------------
def load_2s_audio(path, target_sr=TARGET_SR):
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Audio file not found: {path}")
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    audio = audio.astype(np.float32)
    if len(audio) == 0:
        raise ValueError(f"Empty audio: {path}")
    audio = audio[:MAX_SAMPLES]
    if len(audio) < MAX_SAMPLES:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

@torch.no_grad()
def encode_audio(audio):
    inputs = processor(
        audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(DEVICE)
    attention_mask = inputs.attention_mask.to(DEVICE) if "attention_mask" in inputs else None

    outputs = encoder(
        input_values=input_values,
        attention_mask=attention_mask
    )

    # Mean-pool the frame-level speech representation.
    hidden = outputs.last_hidden_state
    if attention_mask is not None:
        # Wav2Vec2 downsamples time; derive a safe mask for the hidden sequence.
        mask = torch.ones(hidden.shape[:2], device=hidden.device)
    else:
        mask = torch.ones(hidden.shape[:2], device=hidden.device)

    pooled = (hidden * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp_min(1.0)
    return pooled

@torch.no_grad()
def predict_scores(audio_path):
    audio = load_2s_audio(audio_path)
    pooled = encode_audio(audio)
    confidence, assertiveness = head(pooled)

    c = float(confidence.item())
    a = float(assertiveness.item())

    # Human-readable decision. This threshold is a product choice, not a calibrated truth.
    label = "CONFIDENT" if c >= 0.50 else "NOT_CONFIDENT"

    return {
        "label": label,
        "confidence": round(c * 100, 2),
        "assertiveness": round(a * 100, 2)
    }

# ---------------------------------------------------------------------
# 3. CREATE / LOAD A BASELINE TRAINING SET
# ---------------------------------------------------------------------
# Preferred: put your own labeled 2-second clips in:
#   ./data/confidence_labels.csv
#
# CSV columns:
#   audio_path,confidence_score,assertiveness_score
#
# Scores must be in [0,1].
#
# Example:
#   data/confident/clip001.wav,0.92,0.88
#   data/not_confident/clip002.wav,0.18,0.25
#
# If the CSV is absent, the notebook creates a tiny demonstration set
# from any WAV files found under ./data/demo_audio. That demo set is NOT
# suitable for training; it only proves that the pipeline works.

LABEL_CSV = Path("./data/confidence_labels.csv")
DEMO_DIR = Path("./data/demo_audio")

if LABEL_CSV.exists():
    df = pd.read_csv(LABEL_CSV)
    required = {"audio_path", "confidence_score", "assertiveness_score"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing columns: {sorted(missing)}")

    df["audio_path"] = df["audio_path"].astype(str)
    df["confidence_score"] = df["confidence_score"].astype(float).clip(0, 1)
    df["assertiveness_score"] = df["assertiveness_score"].astype(float).clip(0, 1)

    df = df[df["audio_path"].map(os.path.exists)].reset_index(drop=True)
    if len(df) == 0:
        raise ValueError("No existing audio files were found in confidence_labels.csv.")
    print(f"Loaded {len(df)} labeled clips.")
else:
    DEMO_DIR.mkdir(parents=True, exist_ok=True)
    demo_files = sorted(DEMO_DIR.glob("*.wav"))
    if not demo_files:
        print(
            "\nNo confidence_labels.csv was found and ./data/demo_audio is empty.\n"
            "The BASE model will still be saved, and you can run inference after "
            "placing a WAV file in ./data/demo_audio or supplying your own path."
        )
        df = pd.DataFrame(columns=["audio_path", "confidence_score", "assertiveness_score"])
    else:
        # Demonstration only: no legitimate labels can be inferred from unlabeled audio.
        # We deliberately do NOT invent labels.
        print(
            f"Found {len(demo_files)} demo WAV files, but no labels. "
            "No training will be performed in this cell."
        )
        df = pd.DataFrame(columns=["audio_path", "confidence_score", "assertiveness_score"])

# ---------------------------------------------------------------------
# 4. OPTIONAL BASELINE TRAINING ON YOUR LABELED DATA
# ---------------------------------------------------------------------
# This is intentionally lightweight. The point of CELL 1 is to create a
# reproducible BASE checkpoint before the later fine-tuning cell.
#
# If no labels exist, the randomly initialized heads are saved as a valid
# baseline model and can still produce scores, but those scores are NOT
# meaningful until trained.

class LabeledAudioDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        audio = load_2s_audio(row.audio_path)
        return (
            torch.tensor(audio, dtype=torch.float32),
            torch.tensor(float(row.confidence_score), dtype=torch.float32),
            torch.tensor(float(row.assertiveness_score), dtype=torch.float32),
        )

def train_heads_only(frame, epochs=3, batch_size=4, lr=2e-4):
    if len(frame) < 2:
        print("Not enough labeled samples for baseline training; saving untrained baseline head.")
        return

    train_df, val_df = train_test_split(frame, test_size=0.2, random_state=SEED)
    train_ds = LabeledAudioDataset(train_df)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    # Freeze the speech encoder in the BASE training stage.
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad = False

    head.train()
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        losses = []
        for audio_batch, c_target, a_target in train_loader:
            pooled_list = []
            for audio in audio_batch:
                pooled_list.append(encode_audio(audio.numpy()).squeeze(0))
            pooled = torch.stack(pooled_list)

            c_pred, a_pred = head(pooled)
            loss = loss_fn(c_pred, c_target.to(DEVICE)) + loss_fn(a_pred, a_target.to(DEVICE))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        print(f"baseline epoch {epoch+1}/{epochs} loss={np.mean(losses):.4f}")

if len(df) >= 2:
    train_heads_only(df)

# ---------------------------------------------------------------------
# 5. SAVE THE BASE MODEL — CELL 1 MUST END HERE
# ---------------------------------------------------------------------
BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

encoder.save_pretrained(BASE_SAVE_DIR / "encoder")
processor.save_pretrained(BASE_SAVE_DIR / "processor")
torch.save(head.state_dict(), BASE_SAVE_DIR / "confidence_head.pt")

with open(BASE_SAVE_DIR / "config.json", "w") as f:
    json.dump({
        "encoder_name": BASE_MODEL_NAME,
        "sampling_rate": TARGET_SR,
        "max_seconds": MAX_SECONDS,
        "labels": ["CONFIDENT", "NOT_CONFIDENT"],
        "secondary_score": "ASSERTIVENESS",
        "confidence_definition": "continuous speech confidence proxy",
        "assertiveness_definition": "continuous speech assertiveness proxy",
        "proxy_note": "If trained from MSP-Podcast dominance, dominance is used as a proxy and is not identical to confidence."
    }, f, indent=2)

# Smoke-test the saved baseline on the first available WAV, if one exists.
test_candidates = []
if LABEL_CSV.exists() and len(df):
    test_candidates.append(df.iloc[0].audio_path)
test_candidates.extend([str(p) for p in DEMO_DIR.glob("*.wav")])

if test_candidates:
    base_output = predict_scores(test_candidates[0])
    print("\nBASE MODEL OUTPUT")
    print(json.dumps(base_output, indent=2))
else:
    print(
        "\nBASE MODEL SAVED successfully.\n"
        "Add a labeled CSV and WAV files before expecting meaningful confidence scores."
    )

print("\nSaved BASE model to:", BASE_SAVE_DIR.resolve())


In [ ]:

# CELL 2 — FINE-TUNING: fine-tune the BASE model end-to-end and save the FINE-TUNED model
#
# This entire cell performs:
#   1) load the saved BASE checkpoint
#   2) load confidence/assertiveness labels
#   3) fine-tune the wav2vec2 encoder + both heads
#   4) evaluate on a held-out validation split
#   5) save the FINE-TUNED model
#   6) run a final inference example and print the required output
#
# DATA CONTRACT:
#   ./data/confidence_labels.csv
#
# Required columns:
#   audio_path,confidence_score,assertiveness_score
#
# Scores must be [0,1].
#
# For a real confidence model, these scores should come from human annotation.
# If you are using MSP-Podcast dominance as a proxy:
#   confidence_score = dominance
#   assertiveness_score = dominance
#
# Do not silently relabel emotion categories as confidence categories.
# "HAPPY" is not "CONFIDENT", and "DISGUSTED" is not "NOT_CONFIDENT".

import os, json, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import librosa

from transformers import Wav2Vec2Model, Wav2Vec2Processor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_SAVE_DIR = Path("./models/confidence_base")
FINETUNED_SAVE_DIR = Path("./models/confidence_finetuned")
LABEL_CSV = Path("./data/confidence_labels.csv")

TARGET_SR = 16000
MAX_SECONDS = 2.0
MAX_SAMPLES = int(TARGET_SR * MAX_SECONDS)

if not BASE_SAVE_DIR.exists():
    raise FileNotFoundError(
        f"BASE checkpoint not found at {BASE_SAVE_DIR}. Run CELL 1 first."
    )

if not LABEL_CSV.exists():
    raise FileNotFoundError(
        f"{LABEL_CSV} not found. Create it with columns: "
        "audio_path, confidence_score, assertiveness_score"
    )

df = pd.read_csv(LABEL_CSV)
required = {"audio_path", "confidence_score", "assertiveness_score"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV missing columns: {sorted(missing)}")

df["audio_path"] = df["audio_path"].astype(str)
df["confidence_score"] = df["confidence_score"].astype(float).clip(0, 1)
df["assertiveness_score"] = df["assertiveness_score"].astype(float).clip(0, 1)

df = df[df.audio_path.map(os.path.exists)].reset_index(drop=True)
if len(df) < 10:
    raise ValueError(
        f"Only {len(df)} valid labeled clips found. "
        "Use more labeled examples before fine-tuning."
    )

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED
)

print(f"Train clips: {len(train_df)}")
print(f"Validation clips: {len(val_df)}")
print("Device:", DEVICE)

# ---------------------------------------------------------------------
# 1. LOAD THE BASE CHECKPOINT
# ---------------------------------------------------------------------
processor = Wav2Vec2Processor.from_pretrained(BASE_SAVE_DIR / "processor")
encoder = Wav2Vec2Model.from_pretrained(BASE_SAVE_DIR / "encoder").to(DEVICE)

class ConfidenceHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.dropout = nn.Dropout(0.10)
        self.confidence = nn.Linear(hidden_size, 1)
        self.assertiveness = nn.Linear(hidden_size, 1)

    def forward(self, pooled):
        x = self.dropout(pooled)
        confidence = torch.sigmoid(self.confidence(x)).squeeze(-1)
        assertiveness = torch.sigmoid(self.assertiveness(x)).squeeze(-1)
        return confidence, assertiveness

head = ConfidenceHead(encoder.config.hidden_size).to(DEVICE)
head.load_state_dict(torch.load(BASE_SAVE_DIR / "confidence_head.pt", map_location=DEVICE))

# ---------------------------------------------------------------------
# 2. AUDIO DATASET
# ---------------------------------------------------------------------
def load_2s_audio(path):
    audio, sr = librosa.load(str(path), sr=TARGET_SR, mono=True)
    audio = audio.astype(np.float32)
    if len(audio) == 0:
        raise ValueError(f"Empty audio: {path}")
    audio = audio[:MAX_SAMPLES]
    if len(audio) < MAX_SAMPLES:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

class LabeledAudioDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        return (
            torch.tensor(load_2s_audio(row.audio_path), dtype=torch.float32),
            torch.tensor(float(row.confidence_score), dtype=torch.float32),
            torch.tensor(float(row.assertiveness_score), dtype=torch.float32),
        )

train_loader = DataLoader(
    LabeledAudioDataset(train_df),
    batch_size=4,
    shuffle=True
)
val_loader = DataLoader(
    LabeledAudioDataset(val_df),
    batch_size=4,
    shuffle=False
)

# ---------------------------------------------------------------------
# 3. END-TO-END FINE-TUNING
# ---------------------------------------------------------------------
def encode_batch(audio_batch):
    # Processor works with a list of numpy arrays.
    arrays = [x.numpy() for x in audio_batch]
    inputs = processor(
        arrays,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(DEVICE)
    attention_mask = inputs.attention_mask.to(DEVICE) if "attention_mask" in inputs else None

    outputs = encoder(
        input_values=input_values,
        attention_mask=attention_mask
    )

    hidden = outputs.last_hidden_state

    # For fixed 2-second clips, a simple mean pool is sufficient.
    pooled = hidden.mean(dim=1)
    return pooled

optimizer = torch.optim.AdamW(
    [
        {"params": encoder.parameters(), "lr": 1e-5},
        {"params": head.parameters(), "lr": 5e-5},
    ],
    weight_decay=0.01
)

loss_fn = nn.MSELoss()

EPOCHS = 5
best_val_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    encoder.train()
    head.train()
    train_losses = []

    for audio_batch, c_target, a_target in train_loader:
        pooled = encode_batch(audio_batch)

        c_pred, a_pred = head(pooled)

        # Confidence is the primary task: weight it more heavily.
        confidence_loss = loss_fn(c_pred, c_target.to(DEVICE))
        assertiveness_loss = loss_fn(a_pred, a_target.to(DEVICE))

        loss = 2.0 * confidence_loss + 0.5 * assertiveness_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(head.parameters()),
            max_norm=1.0
        )
        optimizer.step()

        train_losses.append(float(loss.item()))

    # -------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------
    encoder.eval()
    head.eval()
    val_losses = []
    c_true, c_pred_all = [], []
    a_true, a_pred_all = [], []

    with torch.no_grad():
        for audio_batch, c_target, a_target in val_loader:
            pooled = encode_batch(audio_batch)
            c_pred, a_pred = head(pooled)

            confidence_loss = loss_fn(c_pred, c_target.to(DEVICE))
            assertiveness_loss = loss_fn(a_pred, a_target.to(DEVICE))
            loss = 2.0 * confidence_loss + 0.5 * assertiveness_loss

            val_losses.append(float(loss.item()))
            c_true.extend(c_target.numpy().tolist())
            c_pred_all.extend(c_pred.cpu().numpy().tolist())
            a_true.extend(a_target.numpy().tolist())
            a_pred_all.extend(a_pred.cpu().numpy().tolist())

    val_loss = float(np.mean(val_losses))
    c_mae = mean_absolute_error(c_true, c_pred_all)
    a_mae = mean_absolute_error(a_true, a_pred_all)

    print(
        f"epoch {epoch+1}/{EPOCHS} | "
        f"train={np.mean(train_losses):.4f} | "
        f"val={val_loss:.4f} | "
        f"confidence_MAE={c_mae:.4f} | "
        f"assertiveness_MAE={a_mae:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            "encoder": {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()},
            "head": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
        }

if best_state is None:
    raise RuntimeError("Fine-tuning did not produce a checkpoint.")

encoder.load_state_dict(best_state["encoder"])
head.load_state_dict(best_state["head"])
encoder.to(DEVICE)
head.to(DEVICE)

# ---------------------------------------------------------------------
# 4. VALIDATION REPORT
# ---------------------------------------------------------------------
encoder.eval()
head.eval()

c_true, c_pred_all = [], []
a_true, a_pred_all = [], []

with torch.no_grad():
    for audio_batch, c_target, a_target in val_loader:
        pooled = encode_batch(audio_batch)
        c_pred, a_pred = head(pooled)

        c_true.extend(c_target.numpy().tolist())
        c_pred_all.extend(c_pred.cpu().numpy().tolist())
        a_true.extend(a_target.numpy().tolist())
        a_pred_all.extend(a_pred.cpu().numpy().tolist())

def safe_r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred)) if len(set(np.round(y_true, 8))) > 1 else None

metrics = {
    "confidence_mae": float(mean_absolute_error(c_true, c_pred_all)),
    "confidence_rmse": float(mean_squared_error(c_true, c_pred_all) ** 0.5),
    "confidence_r2": safe_r2(c_true, c_pred_all),
    "assertiveness_mae": float(mean_absolute_error(a_true, a_pred_all)),
    "assertiveness_rmse": float(mean_squared_error(a_true, a_pred_all) ** 0.5),
    "assertiveness_r2": safe_r2(a_true, a_pred_all),
}

print("\nVALIDATION METRICS")
print(json.dumps(metrics, indent=2))

# ---------------------------------------------------------------------
# 5. SAVE THE FINE-TUNED MODEL
# ---------------------------------------------------------------------
FINETUNED_SAVE_DIR.mkdir(parents=True, exist_ok=True)

encoder.save_pretrained(FINETUNED_SAVE_DIR / "encoder")
processor.save_pretrained(FINETUNED_SAVE_DIR / "processor")
torch.save(head.state_dict(), FINETUNED_SAVE_DIR / "confidence_head.pt")

with open(FINETUNED_SAVE_DIR / "config.json", "w") as f:
    json.dump({
        "encoder_base": "facebook/wav2vec2-base",
        "task": "speech confidence regression",
        "primary_label": "CONFIDENT",
        "secondary_label": "ASSERTIVE",
        "output_range": [0, 100],
        "decision_rule": "CONFIDENT if confidence >= 50%",
        "confidence_proxy_note": "Use human confidence labels for a true confidence model. MSP-Podcast dominance can be used as a proxy but is not identical to confidence.",
        "metrics": metrics
    }, f, indent=2)

# ---------------------------------------------------------------------
# 6. FINAL INFERENCE FUNCTION
# ---------------------------------------------------------------------
@torch.no_grad()
def predict_finetuned(audio_path):
    audio = load_2s_audio(audio_path)
    inputs = processor(
        audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(DEVICE)
    attention_mask = inputs.attention_mask.to(DEVICE) if "attention_mask" in inputs else None

    hidden = encoder(
        input_values=input_values,
        attention_mask=attention_mask
    ).last_hidden_state

    pooled = hidden.mean(dim=1)
    confidence, assertiveness = head(pooled)

    c = float(confidence.item()) * 100.0
    a = float(assertiveness.item()) * 100.0

    return {
        "label": "CONFIDENT" if c >= 50.0 else "NOT_CONFIDENT",
        "confidence": round(c, 2),
        "assertiveness": round(a, 2)
    }

# ---------------------------------------------------------------------
# 7. RUN A REAL EXAMPLE
# ---------------------------------------------------------------------
example_audio = None

if len(val_df):
    example_audio = val_df.iloc[0]["audio_path"]
else:
    wavs = list(Path("./data").rglob("*.wav"))
    if wavs:
        example_audio = str(wavs[0])

if example_audio:
    final_output = predict_finetuned(example_audio)
    print("\nFINE-TUNED MODEL OUTPUT")
    print(json.dumps(final_output, indent=2))
    print("\nExample:", example_audio)
else:
    print("\nFine-tuned model saved, but no WAV file was available for final inference.")

print("\nFINE-TUNED MODEL SAVED TO:")
print(FINETUNED_SAVE_DIR.resolve())
